# Redis (Enterprise AI System Design)

Redis is one of the most frequently asked topics in **EPAM, Microsoft, Amazon, Deloitte, Cognizant, Accenture, and TCS** interviews because almost every production AI application uses it.

---

# 1. What is Redis?

## Definition

**Redis (Remote Dictionary Server)** is an **in-memory NoSQL key-value database** used for **caching, session storage, rate limiting, distributed locking, and message brokering**.

Unlike PostgreSQL, Redis stores data primarily in **RAM**, making it extremely fast.

---

## Interview Answer

> "Redis is an in-memory key-value data store used to improve application performance by caching frequently accessed data, managing sessions, implementing rate limiting, and reducing expensive database or LLM calls."

---

# 2. Why Redis?

Suppose your chatbot asks Bedrock the same question repeatedly.

Without Redis:

```text
User
 │
 ▼
FastAPI
 │
 ▼
AWS Bedrock
 │
 ▼
Answer
```

Every request:

- Calls Bedrock
- Uses tokens
- Increases latency
- Increases cost

---

With Redis:

```text
User
 │
 ▼
FastAPI
 │
 ▼
Redis
 │
 ├── Found
 │      │
 │      ▼
 │   Return Response
 │
 └── Not Found
        │
        ▼
     LangGraph
        │
        ▼
     AWS Bedrock
        │
        ▼
 Save Response in Redis
        │
        ▼
      Return
```

---

# 3. Azure vs AWS

| Azure | AWS |
|--------|-----|
| Azure Cache for Redis | Amazon ElastiCache for Redis |

---

# 4. Where Redis Fits

```text
                       User
                         │
                         ▼
     Azure Front Door / Route53 + CloudFront
                         │
                         ▼
 Azure API Management / Amazon API Gateway
                         │
                         ▼
 Azure App Gateway / AWS ALB
                         │
                         ▼
 FastAPI (Container Apps) / ECS Fargate
                         │
        ┌────────────────┴───────────────┐
        ▼                                ▼
     Redis Cache                  LangGraph
        │                                │
        ▼                                ▼
 Cached Response              Retriever
                                        │
                                        ▼
                 Azure AI Search / Qdrant / OpenSearch
                                        │
                                        ▼
                 Azure OpenAI / AWS Bedrock
```

---

# 5. Internal Working

Redis stores data as

```text
Key → Value
```

Example

```text
Key:
"What is leave policy?"

↓

Value:
"Employees receive 20 annual leave days."
```

Next time

Same Question

↓

Redis

↓

Instant Response

---

# 6. Production Use Cases

## 1. LLM Response Cache ⭐⭐⭐⭐⭐

Most common.

```text
Question

↓

Redis?

↓

Yes

↓

Return

↓

No

↓

LLM

↓

Redis

↓

Return
```

---

## 2. Embedding Cache

Instead of generating embeddings repeatedly

```text
Document

↓

Embedding

↓

Redis
```

---

## 3. Session Storage

Store

- Login
- User Session
- Temporary Context

Instead of

Application Memory.

---

## 4. Rate Limiting

Example

100 requests/minute.

Redis

↓

Tracks

↓

Blocks

After limit.

---

## 5. Chat Memory

Short-term conversation memory.

Example

```text
User

↓

Redis

↓

Previous Messages
```

Long-term history

↓

PostgreSQL.

---

## 6. Background Job Queue

Redis

↓

Celery

↓

Worker

↓

Embedding

↓

Qdrant

---

# 7. Coding Example

Install

```bash
pip install redis
```

Connect

```python
import redis

r = redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True
)
```

---

## Store

```python
r.set("user", "Suraj")

print(r.get("user"))
```

Output

```text
Suraj
```

---

## LLM Cache Example

```python
import redis

cache = redis.Redis(host="localhost", port=6379, decode_responses=True)

question = "What is RAG?"

answer = cache.get(question)

if answer:
    print("From Redis:", answer)
else:
    answer = "RAG stands for Retrieval-Augmented Generation."
    cache.set(question, answer, ex=3600)  # Cache for 1 hour
    print("From LLM:", answer)
```

---

# 8. LangGraph + Redis

```text
Question

↓

Redis

↓

Found?

↓

Yes

↓

Return

↓

No

↓

LangGraph

↓

Bedrock

↓

Save Redis

↓

Return
```

This reduces

- Cost
- Latency
- API calls

---

# 9. Redis vs PostgreSQL

| Redis | PostgreSQL |
|--------|------------|
| RAM | Disk |
| Milliseconds | Slower |
| Cache | Permanent Storage |
| Temporary | Long-term |
| Session | User Data |
| Key-Value | Relational |

---

# 10. Redis vs Qdrant

| Redis | Qdrant |
|--------|---------|
| Key-Value | Vector Database |
| Cache | Embeddings |
| Sessions | Similarity Search |
| Strings | Vectors |

---

# 11. Redis vs S3

| Redis | S3 |
|--------|----|
| RAM | Object Storage |
| Fast | Durable |
| Cache | Files |
| Temporary | Permanent |

---

# 12. Redis Data Structures

| Type | Use Case |
|------|----------|
| String | Cache |
| List | Queue |
| Set | Unique Values |
| Hash | User Profile |
| Sorted Set | Ranking |
| Stream | Event Streaming |

---

# 13. Redis TTL

TTL

=

Time To Live

Example

```python
cache.set(
    "answer",
    "Hello",
    ex=300
)
```

Automatically deleted

after

5 minutes.

---

# 14. Best Practices

✅ Cache LLM responses

✅ Cache embeddings

✅ Store sessions

✅ Use TTL

✅ Never store permanent business data

✅ Use Redis Cluster in production

---

# 15. Common Mistakes

❌ Store PDFs

❌ Store permanent user records

❌ No TTL

❌ Large objects

❌ Depend on Redis as the only source of truth

---

# 16. Interview Questions

### Q1. Why Redis?

To improve performance by caching frequently accessed data and reducing repeated database or LLM calls.

---

### Q2. Why not PostgreSQL?

PostgreSQL is optimized for durable, relational storage.

Redis is optimized for very fast reads and writes.

---

### Q3. Why Redis in AI?

- Cache LLM responses
- Cache embeddings
- Store sessions
- Reduce token cost

---

### Q4. Redis or Vector DB?

Redis

↓

Cache

Vector DB

↓

Similarity Search

---

### Q5. Can Redis store vectors?

Yes, Redis supports vector search through Redis Stack/RediSearch.

However, for enterprise RAG, dedicated vector databases like **Qdrant**, **Pinecone**, or **Amazon OpenSearch** are more commonly chosen because they provide richer vector indexing, filtering, and retrieval capabilities.

---

### Q6. Should chat history be stored in Redis?

Short-term

Yes.

Long-term

No.

Use

- PostgreSQL
- DynamoDB
- Azure SQL

---

# 17. Production Flow

```text
User
 │
 ▼
FastAPI
 │
 ▼
Redis
 │
 ├── Hit
 │      │
 │      ▼
 │   Return Cached Answer
 │
 └── Miss
        │
        ▼
LangGraph
        │
        ▼
Retriever
        │
        ▼
Qdrant
        │
        ▼
AWS Bedrock / Azure OpenAI
        │
        ▼
Save Redis (TTL)
        │
        ▼
Return Response
```

---

# 18. Scenario-Based Question

### Interviewer

> Your chatbot receives the same HR question 10,000 times every day. How would you optimize it?

### Expected Answer

- Check Redis first.
- If cached, return immediately.
- If not cached, invoke LangGraph and Bedrock.
- Store the generated response in Redis with a suitable TTL.
- This reduces latency, token consumption, and infrastructure cost.

---

# 19. EPAM Senior Answer (2 Minutes)

> "In enterprise AI applications, I primarily use Redis as a high-performance cache rather than a primary database. Before invoking the LangGraph workflow and AWS Bedrock or Azure OpenAI, the application checks Redis for an existing response. If a cached result is available, it is returned immediately, avoiding unnecessary LLM inference, reducing latency, and lowering token costs. Redis is also used for session management, rate limiting, embedding caching, and short-term conversational memory. Long-term business data such as users, chat history, and metadata is stored in PostgreSQL or another persistent database, while vector embeddings remain in Qdrant, OpenSearch, or Azure AI Search. This separation ensures high performance, scalability, and maintainability in production AI systems."